# Chess AI — PGN + ZST Training v6 (fix disk space)

**v4 bi overfit tu epoch 3** (best chi roi vao epoch 2/20, Val loss tang dan sau do
trong khi Train loss/acc van tiep tuc cai thien — dau hieu hoc-vet vi data qua it
so voi 11M tham so cua model). Cac fix trong v5:

- `MIN_ELO = 1800` (giam tu 2000) — tang luong data, truoc day loc 2000 chi giu **3.1%** so van.
- `DROPOUT = 0.3` va `WEIGHT_DECAY = 1e-3` (tang tu 0.1 / 1e-4) — regularize manh hon.
- **Early stopping** (`EARLY_STOP_PATIENCE = 3`) — tu dung khi Val khong cai thien, khoi train het 20-30 epoch vo ich.
- `ReduceLROnPlateau` thay cho `CosineAnnealingLR` co dinh — giam LR theo Val loss thuc te.
- Fix bug cache shard: doi `MIN_ELO` **hoac** them/doi dataset PGN moi (qua Add Data) ma khong xoa shard cu se gio **khong** bi resume nham data/loc cu nua — manifest gio luu ca danh sach file PGN da dung, doi 1 trong 2 (bo loc hoac danh sach file) deu kich hoat parse lai.

Tat ca cau hinh chinh tai **Cell 1**.

---

## v6 — fix disk space (loi "tried to use more disk space than is available")

Nguyen nhan: Cell 2 (cu) giai nen `.pgn.zst` -> `.pgn` ra `/kaggle/working`
(ban giai nen co the lon hon ban nen rat nhieu), **cong voi** Cell 3 (cu) ghi
shard bang `np.savez` khong nen, dung `float16` (~2566 bytes/sample). Hai thu
nay cong lai vuot qua dung luong dia Kaggle cho phep.

Fix:
- **Cell 1b (moi)** — don sach `pgn_decompressed/` va shard dinh dang cu con
  sot lai tu lan chay truoc bi loi. Chay 1 lan, an toan de chay lai nhieu lan.
- **Cell 2** — doc thang tu `.pgn.zst` qua `zstandard.stream_reader()`,
  **khong con ghi file `.pgn` giai nen ra dia nua**.
- **Cell 3** — `encode_board()` doi `float16` -> `uint8` (gia tri van la 0/1,
  khong mat gi) + `np.savez_compressed` thay vi `np.savez`. Board planes rat
  thua nen nen rat hieu qua: do tren du lieu test, giam **~35-100 lan** dung
  luong dia, doc shard luc train con **nhanh hon** vi it I/O hon.
- **Cell 6** — `epoch_*.pth` (checkpoint moi epoch) gio tu dong xoa bot, chi
  giu `KEEP_LAST_N_CHECKPOINTS` file gan nhat (cau hinh o Cell 1).


In [ ]:
# Cell 1: Cai dat & Cau hinh
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "chess", "zstandard", "-q"])
from pathlib import Path

# ================================================================
#  CAU HINH CHAT LUONG DU LIEU
# ================================================================

# --- Loc nguoi choi ---
MIN_ELO         = 1600   # FIX overfitting: ha tu 2000 -> 1800 de tang luong data.
                          # 2000 chi giu 3.1% so van (47,828/1,567,015), qua it so
                          # voi model 11M tham so -> hoc vet sau 2 epoch.
                          # Antichess Lichess: 1800+ van la nguoi choi kha manh.

# --- Loc game ---
MIN_GAME_MOVES  = 5      # Loai game < 5 nuoc (abandon/ngat mang/bug)
MAX_GAME_MOVES  = 200    # Loai game > 200 nuoc (bat thuong, rat hiem trong antichess)

# --- Loai time control qua nhanh (bullet 1+0 co nhieu nuoc sai) ---
# None = khong loc time control
# ["180+0","180+2","300+0","300+3"] = chi lay blitz/rapid
MIN_TIME_SECONDS = 180   # Bo qua game co base time < 3 phut (bullet)
                          # Set None neu muon giu ca bullet

# --- So luong game ---
GAMES_PER_FILE  = None   # None = lay tat ca game du dieu kien moi file
MAX_TOTAL_GAMES = None   # None = khong gioi han tong

# ================================================================
#  CAU HINH TRAIN
# ================================================================
EPOCHS           = 30    # FIX: tang tran tu 20 -> 30. Early stopping (xem duoi)
                          # se tu dung som hon neu val khong cai thien, day chi
                          # la gioi han an toan phong khi data moi (ELO thap hon)
                          # can nhieu epoch hon truoc khi bat dau overfit.
LR               = 5e-4  # LR khoi diem (scheduler se tu giam, xem Cell 6)

# --- FIX overfitting: regularization manh hon ---
DROPOUT          = 0.3   # Tang tu 0.1 -> 0.3 trong moi residual block.
WEIGHT_DECAY     = 1e-3  # Tang tu 1e-4 -> 1e-3 trong AdamW.

# --- FIX overfitting: dung dung luc thay vi train het 20/30 epoch ---
EARLY_STOP_PATIENCE = 3  # Dung neu Val combined loss khong cai thien sau 3 epoch lien tiep.
                          # (xem epoch 2 = best, epoch 3-20 chi te hon -> patience=3 se
                          #  dung quanh epoch 5 thay vi chay het 20 epoch vo ich)

BATCH_SIZE       = 512

# KHONG DOI: phai khop voi kDrawValue trong mcts.cpp
DRAW_VALUE       = -0.4

# ================================================================
#  THAM SO KIEN TRUC (KHONG DOI)
# ================================================================
NUM_ACTIONS      = 4288
NUM_INPUT_PLANES = 20
EXISTING_MODEL   = None   # hoac "/kaggle/input/.../best_model_traced.pt"
INPUT_DIR        = "/kaggle/input"

OUTPUT_DIR     = Path("/kaggle/working/chess_pgn_train")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
SHARD_DIR      = OUTPUT_DIR / "shards"
for d in [OUTPUT_DIR, CHECKPOINT_DIR, SHARD_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# FIX disk ("tried to use more disk space than is available"): bo DECOMP_DIR.
# Cell 2 moi doc thang tu .pgn.zst (streaming), khong giai nen ra file rieng nua.
KEEP_LAST_N_CHECKPOINTS = 2   # FIX disk: epoch_*.pth tich luy moi epoch (~vai
                              # tram MB/file vi co ca optimizer state) neu khong
                              # don. Chi giu N file gan nhat de resume (xem Cell 6).
                              # best_model.pth / best_model_traced.pt luon duoc
                              # giu rieng, khong bi anh huong boi gioi han nay.

print("=" * 50)
print(f"MIN_ELO         = {MIN_ELO}")
print(f"MIN_GAME_MOVES  = {MIN_GAME_MOVES}")
print(f"MAX_GAME_MOVES  = {MAX_GAME_MOVES}")
print(f"MIN_TIME_SECONDS= {MIN_TIME_SECONDS}")
print(f"GAMES_PER_FILE  = {GAMES_PER_FILE}")
print(f"EPOCHS          = {EPOCHS}  LR = {LR}")
print(f"DROPOUT         = {DROPOUT}  WEIGHT_DECAY = {WEIGHT_DECAY}")
print(f"EARLY_STOP_PATIENCE = {EARLY_STOP_PATIENCE}")
print(f"KEEP_LAST_N_CHECKPOINTS = {KEEP_LAST_N_CHECKPOINTS}")
print("=" * 50)
print("Cell 1 done")


In [ ]:
# Cell 1b: Don dep dia tu lan chay truoc — CHAY CELL NAY TRUOC khi sang Cell 2
#
# Phien Kaggle hien tai da day dia vi pipeline cu de lai 2 thu:
#   1. pgn_decompressed/ — PGN da giai nen tu .zst (Cell 2 cu ghi ra day, co
#      the gap 8-15 lan dung luong file .zst nen). Cell 2 moi (ben duoi) doc
#      thang tu .zst (streaming) nen KHONG CAN file nay nua.
#   2. shards/ dinh dang cu — encode_board() cu dung float16 + np.savez KHONG
#      nen, ~2566 bytes/sample. Cell 3 moi dung uint8 + np.savez_compressed,
#      nho hon ~35-100 lan tren cung du lieu. Shard cu se duoc thay bang shard
#      moi nho hon nhieu khi parse lai.
#
# Cell nay AN TOAN de chay lai bat ky luc nao (kiem tra dinh dang truoc khi
# xoa): neu shards/ da o dinh dang moi (co "format_version": 2 trong
# manifest.json) thi se GIU LAI de resume binh thuong, khong xoa nham.
# Khong dung toi /kaggle/input (.zst goc, read-only, khong xoa duoc dau).
import shutil, json as _json
from pathlib import Path

def _dir_size_gb(p):
    p = Path(p)
    if not p.exists():
        return 0.0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9

print("[Cleanup] Dung luong TRUOC khi don:")
print(f"  {OUTPUT_DIR} : {_dir_size_gb(OUTPUT_DIR):.2f} GB (tong)")

old_decomp = OUTPUT_DIR / "pgn_decompressed"
manifest_path = SHARD_DIR / "manifest.json"

is_new_format = False
if manifest_path.exists():
    try:
        with open(manifest_path) as f:
            is_new_format = _json.load(f).get("format_version") == 2
    except Exception:
        is_new_format = False

if old_decomp.exists():
    print(f"  -> XOA {old_decomp} ({_dir_size_gb(old_decomp):.2f} GB) — "
          f"khong can nua, Cell 2 moi doc thang tu .zst")
    shutil.rmtree(old_decomp, ignore_errors=True)
else:
    print(f"  {old_decomp} : khong ton tai, bo qua")

if SHARD_DIR.exists() and any(SHARD_DIR.iterdir()) and not is_new_format:
    print(f"  -> XOA {SHARD_DIR} ({_dir_size_gb(SHARD_DIR):.2f} GB) — "
          f"dinh dang cu (chua nen), se tao lai nho hon nhieu o Cell 3/5")
    shutil.rmtree(SHARD_DIR, ignore_errors=True)
    SHARD_DIR.mkdir(parents=True, exist_ok=True)
elif is_new_format:
    print(f"  {SHARD_DIR} : da la dinh dang moi (nen) — GIU LAI de resume, khong xoa.")
else:
    print(f"  {SHARD_DIR} : rong, bo qua")

print(f"\n[Cleanup] Dung luong SAU khi don: {_dir_size_gb(OUTPUT_DIR):.2f} GB")
print("[Cleanup] Xong — chay tiep Cell 2.")


In [ ]:
# Cell 2: Doc PGN truc tiep tu .pgn / .pgn.zst (streaming) — KHONG giai nen ra dia
#
# FIX disk ("Your notebook tried to use more disk space than is available"):
# Ban goc giai nen tung file .zst -> .pgn roi GHI ra /kaggle/working (DECOMP_DIR).
# Voi PGN, ban giai nen tu .zst thuong lon hon ban nen rat nhieu (PGN la text
# lap di lap lai — header, ky hieu nuoc di — nen zstd nen rat tot), va file nay
# nam tren dia /kaggle/working (co gioi han, ~20GB phan output) suot qua trinh
# chay ma khong bao gio bi xoa -> la 1 trong 2 nguyen nhan chinh gay day dia
# (nguyen nhan con lai la shard chua nen o Cell 3).
#
# Fix: dung zstandard.stream_reader() de doc thang tu .zst dang nen, KHONG bao
# gio tao file .pgn day du tren dia — chess.pgn.read_game() doc tung van mot
# qua 1 stream giai nen "on the fly", giai nen toi dau dung toi do.
import glob, io
from pathlib import Path
import chess.pgn
import zstandard as zstd

def _read_games(stream):
    while True:
        try:
            game = chess.pgn.read_game(stream)
        except Exception:
            continue
        if game is None:
            break
        yield game

def iter_games_from_source(path):
    """Sinh tung van (chess.pgn.Game) tu 1 file .pgn hoac .pgn.zst.
    Voi .zst: giai nen streaming, KHONG ghi file trung gian ra dia."""
    path = str(path)
    if path.endswith(".zst"):
        dctx = zstd.ZstdDecompressor()
        with open(path, "rb") as fh, dctx.stream_reader(fh) as reader:
            text_stream = io.TextIOWrapper(reader, encoding="utf-8", errors="replace")
            yield from _read_games(text_stream)
    else:
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            yield from _read_games(f)

# Tim tat ca file nguon: ca .pgn.zst (nen) lan .pgn (da giai nen san, neu co)
# nam trong /kaggle/input — KHONG con buoc giai nen rieng nua.
SOURCE_FILES = sorted(
    glob.glob(f"{INPUT_DIR}/**/*.pgn.zst", recursive=True) +
    glob.glob(f"{INPUT_DIR}/**/*.pgn", recursive=True)
)
if not SOURCE_FILES:
    raise FileNotFoundError(f"Khong tim thay .pgn hoac .pgn.zst trong {INPUT_DIR}")

total_mb = sum(Path(p).stat().st_size for p in SOURCE_FILES) / 1e6
print(f"[Source] {len(SOURCE_FILES)} file nguon (.pgn / .pgn.zst), tong {total_mb:.0f}MB")
print("[Source] Doc truc tiep (streaming) — khong giai nen ra dia, khong ton dung luong.")
for p in SOURCE_FILES:
    print(f"  {p}  ({Path(p).stat().st_size/1e6:.1f}MB)")
print("Cell 2 done")


In [ ]:
# Cell 3: Board encoding + Shard writer voi bo loc chat luong
# FIX disk (xem Cell 2): encode_board() doi float16 -> uint8 (cac gia tri chi la
# 0/1 nen khong mat thong tin gi) + flush() dung np.savez_compressed thay vi
# np.savez. Du lieu board planes rat thua (hau het la 0) nen nen rat hieu qua:
# do thuc te tren du lieu test, ~2566 bytes/sample -> ~25-70 bytes/sample
# (giam 35-100 lan dung luong dia), doc shard luc train cung NHANH HON vi it
# I/O hon (xem benchmark trong phan giai thich).
import os, glob, time, json as _json, random
import chess, chess.pgn
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[GPU] {device}")
if device.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"[GPU] {p.name}  {p.total_memory/1e9:.1f}GB")

_PIECE_PLANE = {
    (chess.PAWN,   chess.WHITE): 0,  (chess.PAWN,   chess.BLACK): 6,
    (chess.KNIGHT, chess.WHITE): 1,  (chess.KNIGHT, chess.BLACK): 7,
    (chess.BISHOP, chess.WHITE): 2,  (chess.BISHOP, chess.BLACK): 8,
    (chess.ROOK,   chess.WHITE): 3,  (chess.ROOK,   chess.BLACK): 9,
    (chess.QUEEN,  chess.WHITE): 4,  (chess.QUEEN,  chess.BLACK): 10,
    (chess.KING,   chess.WHITE): 5,  (chess.KING,   chess.BLACK): 11,
}
_PROMO_OFFSET = 4096
_PROMO_IDX    = {chess.KNIGHT: 0, chess.BISHOP: 1, chess.ROOK: 2}

def encode_board(board, rep_count=0):
    # FIX disk: uint8 thay vi float16 — gia tri luon la 0 hoac 1 nen khong mat
    # do chinh xac gi, chi giam 1 nua kich thuoc truoc khi nen, va nen (zlib
    # trong np.savez_compressed) hoat dong tot hon tren uint8 thua.
    planes = np.zeros((NUM_INPUT_PLANES, 8, 8), dtype=np.uint8)
    for sq, piece in board.piece_map().items():
        planes[_PIECE_PLANE[(piece.piece_type, piece.color)], sq >> 3, sq & 7] = 1
    if board.turn == chess.WHITE: planes[12] = 1
    planes[13] = int(board.has_kingside_castling_rights(chess.WHITE))
    planes[14] = int(board.has_queenside_castling_rights(chess.WHITE))
    planes[15] = int(board.has_kingside_castling_rights(chess.BLACK))
    planes[16] = int(board.has_queenside_castling_rights(chess.BLACK))
    if board.ep_square is not None:
        planes[17, board.ep_square >> 3, board.ep_square & 7] = 1
    if rep_count >= 1: planes[18] = 1
    if rep_count >= 2: planes[19] = 1
    return planes

def move_to_idx(move):
    if move.promotion and move.promotion != chess.QUEEN:
        idx = _PROMO_IDX.get(move.promotion)
        if idx is None:
            return move.from_square * 64 + move.to_square
        return _PROMO_OFFSET + idx * 64 + move.to_square
    return move.from_square * 64 + move.to_square

def parse_time_control(tc_str):
    """Tra ve base_seconds tu chuoi TimeControl (vd '180+0' -> 180). None neu loi."""
    if not tc_str or tc_str == "-":
        return None
    try:
        return int(tc_str.split("+")[0].split("/")[-1])
    except Exception:
        return None

SHARD_SIZE = 80_000

def parse_pgn_to_shards(source_files, shard_dir, games_per_file=None,
                         max_total_games=None, min_elo=0,
                         min_moves=5, max_moves=200, min_time_seconds=None):
    shard_dir = Path(shard_dir)
    shard_dir.mkdir(parents=True, exist_ok=True)
    manifest  = shard_dir / "manifest.json"

    # FIX disk: dung ten+kich thuoc cua file NGUON (.pgn / .pgn.zst nam trong
    # /kaggle/input, read-only nen khong bao gio mat) de phat hien dataset doi,
    # thay vi can thu muc PGN da giai nen (da bo o Cell 2 — xem giai thich tren).
    src_names = [os.path.basename(p) for p in source_files]
    src_sizes = [Path(p).stat().st_size for p in source_files]

    if manifest.exists():
        with open(manifest) as f:
            info = _json.load(f)
        paths = [str(shard_dir / f"shard_{i:04d}.npz") for i in range(info.get("num_shards", 0))]
        same_filter = (info.get("min_elo") == min_elo and
                       info.get("min_moves") == min_moves and
                       info.get("max_moves") == max_moves and
                       info.get("min_time_seconds") == min_time_seconds and
                       info.get("games_per_file") == games_per_file)
        same_files = (info.get("source_files") == src_names and
                      info.get("source_sizes") == src_sizes)
        # FIX disk: shard cu (truoc fix nay) khong co "format_version" -> luon
        # bi coi la "khac dinh dang", buoc parse lai sang dinh dang nen moi.
        same_format = info.get("format_version") == 2
        if same_filter and same_files and same_format and all(Path(p).exists() for p in paths):
            print(f"[Shards] Resume: {info['num_shards']} shards, "
                  f"{info['total_samples']:,} samples, {info['total_games']:,} games")
            print(f"  (bo loc luc parse: elo>={info.get('min_elo',0)}, "
                  f"moves={info.get('min_moves',0)}-{info.get('max_moves','?')}, "
                  f"min_time={info.get('min_time_seconds','no')})")
            return paths
        else:
            reasons = []
            if not same_filter:
                reasons.append(f"bo loc doi (cu elo>={info.get('min_elo')} -> moi elo>={min_elo})")
            if not same_files:
                old_n = len(info.get("source_files", []))
                reasons.append(f"so file nguon doi ({old_n} -> {len(src_names)}, "
                                f"co the do them/bo dataset)")
            if not same_format:
                reasons.append("shard cu dinh dang chua nen -> parse lai sang dinh dang nen (FIX disk)")
            print(f"[Shards] {'; '.join(reasons)} => parse lai tu dau, bo qua shard cu.")

    print(f"[Shards] {len(source_files)} file nguon (.pgn / .pgn.zst)")
    print(f"  Bo loc: elo>={min_elo}  moves={min_moves}-{max_moves}  "
          f"min_time={f'{min_time_seconds}s' if min_time_seconds else 'tat ca'}")
    print(f"  Gioi han: {f'{games_per_file:,}/file' if games_per_file else 'tat ca'} "
          f"| tong {f'{max_total_games:,}' if max_total_games else 'khong gioi han'}")

    buf_s, buf_m, buf_v = [], [], []
    shard_paths = []
    shard_idx = total_games = total_samples = 0
    stats = {"skip_elo":0, "skip_result":0, "skip_moves":0,
             "skip_time":0, "skip_legal":0, "accepted":0}

    def flush():
        nonlocal shard_idx
        path = shard_dir / f"shard_{shard_idx:04d}.npz"
        # FIX disk: savez_compressed thay vi savez (board planes rat thua -> nen
        # cuc tot). states gio la uint8 (tu encode_board), khong phai float16.
        np.savez_compressed(path,
            states=np.stack(buf_s),
            moves=np.array(buf_m,  dtype=np.int16),
            values=np.array(buf_v, dtype=np.float32))
        print(f"  [Shard {shard_idx:04d}] {len(buf_m):,} samples "
              f"-> {path.stat().st_size/1e6:.1f}MB")
        shard_paths.append(str(path))
        buf_s.clear(); buf_m.clear(); buf_v.clear()
        shard_idx += 1

    for file_no, src_path in enumerate(source_files, 1):
        if max_total_games and total_games >= max_total_games:
            print(f"[Shards] Du max_total_games={max_total_games:,}, dung.")
            break

        size_mb = Path(src_path).stat().st_size / 1e6
        print(f"\n[Parse {file_no}/{len(source_files)}] {os.path.basename(src_path)} ({size_mb:.0f}MB nen)")
        t0 = time.time()
        file_games = 0

        # FIX disk: doc truc tiep tu .pgn/.pgn.zst qua iter_games_from_source()
        # (dinh nghia o Cell 2) — streaming, KHONG qua file .pgn trung gian tren dia.
        for game in iter_games_from_source(src_path):
            if games_per_file and file_games >= games_per_file:
                print(f"  Du {games_per_file:,} games/file, sang file tiep")
                break
            if max_total_games and total_games >= max_total_games:
                break

            hdrs = game.headers

            # --- Loc result ---
            result = hdrs.get("Result", "*")
            if result not in ("1-0", "0-1", "1/2-1/2"):
                stats["skip_result"] += 1; continue

            # --- Loc ELO ---
            if min_elo > 0:
                try:
                    we = int(hdrs.get("WhiteElo", "0") or "0")
                    be = int(hdrs.get("BlackElo",  "0") or "0")
                    if we < min_elo or be < min_elo:
                        stats["skip_elo"] += 1; continue
                except ValueError:
                    stats["skip_elo"] += 1; continue

            # --- Loc time control ---
            if min_time_seconds is not None:
                tc = parse_time_control(hdrs.get("TimeControl", ""))
                if tc is None or tc < min_time_seconds:
                    stats["skip_time"] += 1; continue

            # --- Dem nuoc va encode ---
            board = game.board()
            seen  = {}
            g_s, g_m, g_w = [], [], []
            move_count = 0
            legal_error = False
            for move in game.mainline_moves():
                if not board.is_legal(move):
                    legal_error = True; break
                move_count += 1
                if move_count > max_moves:
                    break
                epd = board.epd()
                rep = seen.get(epd, 0); seen[epd] = rep + 1
                mid = move_to_idx(move)
                if 0 <= mid < NUM_ACTIONS:
                    g_s.append(encode_board(board, rep))
                    g_m.append(mid)
                    g_w.append(board.turn == chess.WHITE)
                board.push(move)

            if legal_error:
                stats["skip_legal"] += 1; continue

            # --- Loc so nuoc ---
            n = len(g_m)
            if n < min_moves:
                stats["skip_moves"] += 1; continue
            if n > max_moves:
                # Giu max_moves nuoc dau, bo phan sau
                g_s = g_s[:max_moves]; g_m = g_m[:max_moves]
                g_w = g_w[:max_moves]; n = max_moves

            vals = np.empty(n, dtype=np.float32)
            if   result == "1-0": vals[:] = [1.0 if w else -1.0 for w in g_w]
            elif result == "0-1": vals[:] = [-1.0 if w else 1.0 for w in g_w]
            else:                 vals[:] = DRAW_VALUE

            buf_s.extend(g_s)
            buf_m.extend(g_m)
            buf_v.extend(vals.tolist())
            file_games    += 1
            total_games   += 1
            total_samples += n
            stats["accepted"] += 1

            if len(buf_m) >= SHARD_SIZE:
                flush()

            if file_games % 25_000 == 0:
                elapsed = time.time() - t0
                accept_rate = stats["accepted"] / max(1, sum(stats.values())) * 100
                print(f"  {file_games:,} games | {total_samples:,} samples | "
                      f"{elapsed/60:.0f}min | accept={accept_rate:.1f}%")

        elapsed = time.time() - t0
        print(f"  Xong: {file_games:,} games chap nhan trong {elapsed/60:.0f}min")

    if buf_m:
        flush()

    total_seen = sum(stats.values())
    print(f"\n[Stats] Tong xem: {total_seen:,}")
    print(f"  Chap nhan : {stats['accepted']:,}  ({stats['accepted']/max(1,total_seen)*100:.1f}%)")
    print(f"  Bo ELO    : {stats['skip_elo']:,}  ({stats['skip_elo']/max(1,total_seen)*100:.1f}%)")
    print(f"  Bo result : {stats['skip_result']:,}")
    print(f"  Bo so nuoc: {stats['skip_moves']:,}")
    print(f"  Bo time   : {stats['skip_time']:,}")
    print(f"  Bo illegal: {stats['skip_legal']:,}")

    with open(manifest, "w") as f:
        _json.dump({"num_shards": shard_idx,
                    "total_samples": total_samples,
                    "total_games": total_games,
                    "min_elo": min_elo,
                    "min_moves": min_moves,
                    "max_moves": max_moves,
                    "min_time_seconds": min_time_seconds,
                    "games_per_file": games_per_file,
                    "source_files": src_names,
                    "source_sizes": src_sizes,
                    "format_version": 2}, f)   # FIX disk: danh dau dinh dang da nen

    print(f"\n[Shards] Done: {shard_idx} shards | {total_samples:,} samples | {total_games:,} games")
    return shard_paths

def load_shard(path):
    data = np.load(path)
    return (data["states"].astype(np.float32),
            data["moves"].astype(np.int32),
            data["values"])

print("Cell 3 done")


In [ ]:
# Cell 4: Model
class ResidualBlock(nn.Module):
    def __init__(self, ch=128, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(ch)
        self.drop  = nn.Dropout2d(p=dropout)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(ch)
    def forward(self, x):
        r = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        return F.relu(self.bn2(self.conv2(out)) + r)

class ChessPolicyNet(nn.Module):
    NUM_INPUT_PLANES = 20
    NUM_ACTIONS      = 4288
    def __init__(self, num_blocks=6, ch=128, pol_ch=32, dropout=0.1):
        super().__init__()
        self.conv_init   = nn.Conv2d(self.NUM_INPUT_PLANES, ch, 3, padding=1, bias=False)
        self.bn_init     = nn.BatchNorm2d(ch)
        self.blocks      = nn.ModuleList([ResidualBlock(ch, dropout) for _ in range(num_blocks)])
        self.policy_conv = nn.Conv2d(ch, pol_ch, 1, bias=False)
        self.policy_bn   = nn.BatchNorm2d(pol_ch)
        self.fc          = nn.Linear(pol_ch * 64, self.NUM_ACTIONS)
        self.value_conv  = nn.Conv2d(ch, 32, 1, bias=False)
        self.value_bn    = nn.BatchNorm2d(32)
        self.value_fc1   = nn.Linear(32 * 64, 256)
        self.value_fc2   = nn.Linear(256, 64)
        self.value_head  = nn.Linear(64, 1)
    def forward(self, x):
        out = F.relu(self.bn_init(self.conv_init(x)))
        for b in self.blocks: out = b(out)
        p = F.relu(self.policy_bn(self.policy_conv(out)))
        policy = self.fc(torch.flatten(p, 1))
        v = F.relu(self.value_bn(self.value_conv(out)))
        v = F.relu(self.value_fc1(torch.flatten(v, 1)))
        v = F.relu(self.value_fc2(v))
        return policy, torch.tanh(self.value_head(v))

def check_value_head(m, tag=""):
    m.eval()
    test_fens = [
        ("Start",      chess.STARTING_FEN),
        ("W+Q+R",      "4k3/8/8/8/8/8/8/4KQR1 w - - 0 1"),
        ("B+Q+R",      "4K3/8/8/8/8/8/8/r1bqk3 b - - 0 1"),
        ("Equal",      "4k3/4p3/8/8/8/8/4P3/4K3 w - - 0 1"),
        ("After 1.e4", "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1"),
    ]
    batch = torch.cat([
        torch.from_numpy(encode_board(chess.Board(fen))).unsqueeze(0)
        for _, fen in test_fens
    ]).float().to(device)  # float16->float32, model weights dùng float32
    with torch.no_grad(): _, vals = m(batch)
    vals = vals.squeeze(1)
    std = vals.std().item()
    status = "OK" if std > 0.05 else "COLLAPSED!"
    print(f"  [ValCheck{tag}] std={std:.3f}  " +
          "  ".join(f"{n}:{v:+.3f}" for (n,_),v in zip(test_fens, vals.tolist())) + f"  {status}")
    m.train()

def load_from_traced(traced_path, model):
    traced    = torch.jit.load(traced_path, map_location="cpu")
    traced_sd = dict(traced.state_dict())
    model_sd  = model.state_dict()
    matched   = {k: v for k, v in traced_sd.items()
                 if k in model_sd and v.shape == model_sd[k].shape}
    model_sd.update(matched)
    model.load_state_dict(model_sd, strict=False)
    print(f"[WeightLoad] Matched {len(matched)}/{len(model_sd)} layers")

def export_ts(m, path):
    raw = getattr(m, "_orig_mod", m); raw.eval().cpu()
    with torch.no_grad():
        traced = torch.jit.trace(raw, torch.zeros(1, NUM_INPUT_PLANES, 8, 8))
    traced.save(str(path)); raw.to(device).train()
    print(f"  [Export] {Path(path).name}  ({Path(path).stat().st_size/1e6:.1f}MB)")

model = ChessPolicyNet(dropout=DROPOUT).to(device)  # FIX: dropout tu Cell 1 (0.1 -> 0.3)
print(f"[Model] {sum(p.numel() for p in model.parameters()):,} params  (dropout={DROPOUT})")
if EXISTING_MODEL and Path(EXISTING_MODEL).exists():
    load_from_traced(EXISTING_MODEL, model)
else:
    print("[Model] Train from scratch")
print("Cell 4 done")



In [ ]:
# Cell 5: Parse PGN -> Shards (voi bo loc tu Cell 1)
shard_paths = parse_pgn_to_shards(
    source_files     = SOURCE_FILES,  # FIX disk: tu Cell 2, khong qua DECOMP_DIR nua
    shard_dir        = SHARD_DIR,
    games_per_file   = GAMES_PER_FILE,
    max_total_games  = MAX_TOTAL_GAMES,
    min_elo          = MIN_ELO,
    min_moves        = MIN_GAME_MOVES,
    max_moves        = MAX_GAME_MOVES,
    min_time_seconds = MIN_TIME_SECONDS,
)

import random
random.seed(42)
shuffled = shard_paths.copy()
random.shuffle(shuffled)
n_val        = max(1, len(shuffled) // 10)
val_shards   = shuffled[:n_val]
train_shards = shuffled[n_val:]

def count_samples(paths):
    n = 0
    for p in paths:
        d = np.load(p)
        n += len(d["moves"])
    return n

n_tr = count_samples(train_shards)
n_vl = count_samples(val_shards)
print(f"[Split] Train: {len(train_shards)} shards / {n_tr:,} samples")
print(f"        Val  : {len(val_shards)} shards / {n_vl:,} samples")
print("Cell 5 done")


In [ ]:
# Cell 6: Training shard-by-shard
POLICY_WEIGHT = 1.0
VALUE_WEIGHT  = 1.5

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
# FIX: CosineAnnealingLR(T_max=EPOCHS) gia dinh train du EPOCHS, nhung gio EPOCHS
# chi la tran an toan (early stopping co the dung som hon nhieu). ReduceLROnPlateau
# phan ung theo Val loss thuc te: giam LR khi val ngung cai thien, khop voi
# EARLY_STOP_PATIENCE thay vi lich giam LR co dinh tu truoc.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=1, min_lr=LR * 0.02)
use_cuda  = device.type == "cuda"
scaler    = torch.amp.GradScaler('cuda', enabled=use_cuda)

best_val    = float("inf")
best_path   = OUTPUT_DIR / "best_model.pth"
traced_path = OUTPUT_DIR / "best_model_traced.pt"
start_epoch = 1
epochs_no_improve = 0   # FIX: dem so epoch lien tiep Val khong cai thien (early stopping)

ckpts = sorted(CHECKPOINT_DIR.glob("epoch_*.pth"),
               key=lambda p: int(p.stem.split("_")[1]) if p.stem.split("_")[1].isdigit() else 0)
if ckpts:
    ck = torch.load(ckpts[-1], map_location=device, weights_only=False)
    model.load_state_dict(ck["model_state_dict"], strict=False)
    optimizer.load_state_dict(ck["optimizer_state_dict"])
    if "scheduler_state_dict" in ck:
        scheduler.load_state_dict(ck["scheduler_state_dict"])
    start_epoch = int(ck["epoch"]) + 1
    best_val    = float(ck.get("best_combined", float("inf")))
    epochs_no_improve = int(ck.get("epochs_no_improve", 0))
    print(f"[Resume] Epoch {start_epoch}  best={best_val:.4f}  "
          f"no_improve={epochs_no_improve}/{EARLY_STOP_PATIENCE}")

def run_shards(shards, train_mode):
    if train_mode:
        model.train()
        order = shards.copy(); random.shuffle(order)
    else:
        model.eval()
        order = shards

    tot_p = tot_v = tot_acc = tot_n = 0
    for shard_path in order:
        s_np, m_np, v_np = load_shard(shard_path)
        ok = (m_np >= 0) & (m_np < NUM_ACTIONS)
        s_np = s_np[ok]; m_np = m_np[ok]; v_np = v_np[ok]

        ds     = TensorDataset(torch.from_numpy(s_np),
                               torch.from_numpy(m_np).long(),
                               torch.from_numpy(v_np))
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=train_mode,
                            drop_last=train_mode, num_workers=0, pin_memory=use_cuda)

        ctx = torch.enable_grad() if train_mode else torch.no_grad()
        with ctx:
            for s, m, v in loader:
                s = s.to(device, non_blocking=True)
                m = m.to(device, non_blocking=True)
                v = v.to(device, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=use_cuda):
                    logits, val_pred = model(s)
                    d_rate = float((v.abs() < 0.5).float().mean())
                    _vw    = VALUE_WEIGHT * (1.5 if d_rate > 0.85 else 1.0)
                    l_pol  = F.cross_entropy(logits, m)
                    l_val  = F.mse_loss(val_pred.squeeze(1), v)
                    loss   = POLICY_WEIGHT * l_pol + _vw * l_val
                if train_mode:
                    optimizer.zero_grad(set_to_none=True)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer); scaler.update()
                bs      = s.size(0)
                tot_acc += (logits.argmax(1) == m).float().mean().item() * bs
                tot_p   += l_pol.item() * bs
                tot_v   += l_val.item() * bs
                tot_n   += bs

        del s_np, m_np, v_np, ds, loader
        if use_cuda: torch.cuda.empty_cache()

    return tot_p / tot_n, tot_v / tot_n, tot_acc / tot_n

print(f"{'='*60}")
print(f"TRAIN epoch {start_epoch}->{EPOCHS}  lr={LR}  batch={BATCH_SIZE}")
print(f"Shards: {len(train_shards)} train / {len(val_shards)} val")
print(f"{'='*60}")
check_value_head(model, " [truoc train]")

for epoch in range(start_epoch, EPOCHS + 1):
    t_ep   = time.time()
    lr_now = optimizer.param_groups[0]["lr"]   # LR dung cho epoch nay
    print(f"\n-- Epoch {epoch:02d}/{EPOCHS} --")
    tr_p, tr_v, tr_acc = run_shards(train_shards, train_mode=True)
    vl_p, vl_v, vl_acc = run_shards(val_shards,   train_mode=False)

    combined = POLICY_WEIGHT * vl_p + VALUE_WEIGHT * vl_v
    # FIX: ReduceLROnPlateau can metric Val (combined) sau khi tinh xong,
    # khac voi CosineAnnealingLR.step() goi vo dieu kien o tren truoc day.
    scheduler.step(combined)
    elapsed  = time.time() - t_ep

    print(f"  [{elapsed/60:.1f}min  lr={lr_now:.2e}]")
    print(f"  Train: pol={tr_p:.4f} val={tr_v:.4f} acc={tr_acc*100:.2f}%")
    print(f"  Val  : pol={vl_p:.4f} val={vl_v:.4f} acc={vl_acc*100:.2f}%  combined={combined:.4f}")
    if epoch % 5 == 0 or epoch == 1:
        check_value_head(model, f" [ep{epoch}]")

    is_best = combined < best_val
    if is_best:
        best_val = combined  # update trước để state lưu giá trị chính xác
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    state = {"epoch": epoch,
             "model_state_dict":     model.state_dict(),
             "optimizer_state_dict": optimizer.state_dict(),
             "scheduler_state_dict": scheduler.state_dict(),
             "val_policy_loss": vl_p, "val_value_loss": vl_v,
             "val_accuracy": vl_acc, "best_combined": best_val,
             "epochs_no_improve": epochs_no_improve}
    torch.save(state, CHECKPOINT_DIR / f"epoch_{epoch:02d}.pth")

    # FIX disk: epoch_*.pth tich luy moi epoch (model+optimizer+scheduler state,
    # ~vai tram MB/file) neu khong don se gop phan gay day dia qua nhieu epoch.
    # Chi giu KEEP_LAST_N_CHECKPOINTS file gan nhat (du de resume); best_path /
    # traced_path la file rieng, KHONG bi anh huong.
    old_ckpts = sorted(CHECKPOINT_DIR.glob("epoch_*.pth"),
                        key=lambda p: int(p.stem.split("_")[1]) if p.stem.split("_")[1].isdigit() else 0)
    for stale in old_ckpts[:-KEEP_LAST_N_CHECKPOINTS]:
        stale.unlink(missing_ok=True)

    if is_best:
        torch.save(state, best_path)
        export_ts(model, traced_path)
        print(f"  BEST (combined={combined:.4f})")
    else:
        print(f"  No improve ({epochs_no_improve}/{EARLY_STOP_PATIENCE})")
    print("-"*60)

    # FIX: Early stopping — epoch 2 la best, epoch 3-20 chi te di o ban goc vi
    # van chay het 20 epoch du khong con cai thien gi (lang phi GPU + qua-fit nang hon).
    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f"\n[EarlyStop] Val khong cai thien sau {EARLY_STOP_PATIENCE} epoch lien tiep "
              f"-> dung tai epoch {epoch} (best vẫn la {best_val:.4f}).")
        break

print(f"\nDone. Best={best_val:.4f} -> {traced_path}")



In [ ]:
# Cell 7: Kiem tra cuoi & export
ck = torch.load(OUTPUT_DIR / "best_model.pth", map_location=device, weights_only=False)
model.load_state_dict(ck["model_state_dict"], strict=False)
print(f"Best: epoch={ck['epoch']}  val_policy={ck['val_policy_loss']:.4f}  "
      f"val_value={ck['val_value_loss']:.4f}  acc={ck['val_accuracy']*100:.2f}%")
check_value_head(model, " [final]")

traced_path = OUTPUT_DIR / "best_model_traced.pt"
print(f"\nOUTPUT : {traced_path}")
if traced_path.exists():
    print(f"SIZE   : {traced_path.stat().st_size/1e6:.1f}MB")
else:
    print(f"[Warning] Chua tim thay {traced_path.name} ")
    print(f"  (neu training moi chay lan dau, file se co sau epoch 1)")
print("\nLuu file nay -> Kaggle Dataset 'chess-model'")
